# 4. Score molecules with a trained model

Scores any SMILES with the models of a finished run, using the inference code in
[../../serving/](../../serving/) — the same code behind the FastAPI service, so these scores match the API
and the benchmark. Replaces the `*_inference*` notebooks' "score my own molecules" use.

It works for both methods: a `finetune` run uses its fine-tuned model, an `embed_lgbm` run embeds the
molecules and scores them with its LightGBM models.

## Settings

In [ ]:
EXPERIMENT = "sample-benchmark"
RUN_NAME = "lgbm_pretrained_fused"     # a finished run folder in the experiment
MODELS = "full"                        # "full" = the model trained on all rows; "folds" = fold models averaged

SMILES = ["CCO", "c1ccccc1O", "CC(=O)Nc1ccc(O)cc1", "CN1C=NC2=C1C(=O)N(C)C(=O)N2C", "not_a_smiles"]

In [ ]:
from pathlib import Path
import sys

import pandas as pd

# Make the benchmark code importable (works from this folder or from the repo root)
BENCHMARK = next(p / rel for p in [Path.cwd(), *Path.cwd().parents] for rel in ("", "aircheck/benchmark")
                 if (p / rel / "methods" / "common.py").exists())
sys.path.insert(0, str(BENCHMARK))
AIRCHECK = BENCHMARK.parent
EXP_DIR = AIRCHECK / "output" / EXPERIMENT
print(f"Benchmark code : {BENCHMARK}")
print(f"Experiment     : {EXP_DIR}")

In [ ]:
import json

RUN_DIR = EXP_DIR / RUN_NAME
assert (RUN_DIR / "run.json").exists(), f"No finished run in {RUN_DIR}"
run_info = json.loads((RUN_DIR / "run.json").read_text())
print(f"{run_info['name']}  ({run_info['method']}, {run_info['created']})")
print(f"settings: {run_info['settings']}")

## Load the model and score

In [ ]:
from methods import common

if run_info["method"] == "finetune":
    source = RUN_DIR / "models" / "final" if MODELS == "full" else RUN_DIR / "models"
    model = common.mv_inference.load_model(source, models=MODELS)
    predict = common.mv_inference.predict
    print(f"MMELON {model.view} model(s) from {source.name} on {model.device}")
else:
    encoder = run_info["settings"].get("encoder") or common.base_model_dir()
    model = common.lgbm_inference.load_model(RUN_DIR, encoder_path=encoder, models=MODELS)
    predict = common.lgbm_inference.predict
    print(f"LightGBM on {model.view} embeddings, {len(model.boosters)} model(s) on {model.device}")

In [ ]:
results = pd.DataFrame(predict(model, SMILES))
results.sort_values("score", ascending=False, na_position="last")

`score` is the model's probability that the molecule is a hit. A SMILES RDKit cannot read gets
`valid = False` and no score.

To score a whole file instead, read it into a list of SMILES and pass that; for thousands of molecules use a
benchmark run (`run.py`) or the API in [../../serving/](../../serving/).